In [3]:
!pip install playwright pandas openpyxl
!playwright install chromium

In [4]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression

# Load dataset (.xlsx)
df = pd.read_excel("/content/dataset_labeled_v6.xlsx")

target_cols = [
    "aspek_kualitas", "aspek_bahan", "aspek_ukuran",
    "aspek_desain", "aspek_warna", "aspek_kenyamanan"
]

# ==============================================================================
# 5.1 DATA CLEANING & 5.3 TEXT PREPROCESSING
# ==============================================================================
# 1. Handling Missing Values
df = df.dropna(subset=["ulasan"]).copy()
df[target_cols] = df[target_cols].fillna("netral")

# 2. Removing Duplicates
df = df.drop_duplicates(subset=["ulasan"]).copy()

# 3. Data Type Conversion
df["ulasan"] = df["ulasan"].astype(str)
for col in target_cols:
    df[col] = df[col].astype(str)

# 4. Pembersihan Noise, Template Ulasan, Lowercasing & Preserving Negation Words
def preprocess_text(text):
    text = text.lower()  # Lowercasing
    # Hapus awalan template e-commerce
    text = re.sub(
        r"\b(harga|styling|tampilan|saran tampilan|kualitas|warna|bahan|ukuran|kenyamanan)\s*:",
        " ",
        text,
    )
    # Hapus tanda baca, angka, dan emoji (kata negasi seperti 'tidak', 'kurang', 'bukan' tetap dipertahankan)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["ulasan_clean"] = df["ulasan"].apply(preprocess_text)

# 5. Hapus ulasan kosong maupun ulasan pendek yang hanya 1 sampai 2 kata (simpan minimal 3 kata)
df = df[df["ulasan_clean"].str.split().str.len() > 2].reset_index(drop=True)

# Hitung sisa data tanpa aspek (memiliki_label == False / jumlah_aspek == 0)
tanpa_aspek = (df[target_cols].apply(lambda x: x.str.lower()) == "netral").all(axis=1).sum()

print("=== 5.1 DATA CLEANING ===")
print(f"Total data bersih        : {len(df)} baris")
print(f"Data tanpa aspek (netral): {tanpa_aspek} baris")
print("Contoh hasil pembersihan teks:")
print(df[["ulasan", "ulasan_clean"]].head(3), "\n")

# ==============================================================================
# 5.2 FEATURE ENGINEERING (FEATURE SELECTION) & 5.3 TARGET ENCODING
# ==============================================================================
# Buang kolom metadata bocor (memiliki_label, jumlah_aspek, label_json, dll.)
X = df["ulasan_clean"]

# Target Encoding: negatif -> 0, netral -> 1, positif -> 2
label_map = {"negatif": 0, "netral": 1, "positif": 2}
y = df[target_cols].apply(lambda col: col.str.strip().str.lower().map(label_map)).astype(int)

print("=== 5.2 & 5.3 FEATURE SELECTION & TARGET ENCODING ===")
print(f"Fitur Input (X)  : {X.name} (shape: {X.shape})")
print(f"Target Output (Y): {list(y.columns)} (shape: {y.shape})")
print(y.head(3), "\n")

# ==============================================================================
# 5.4 DATA SPLITTING (80% TRAIN, 20% TEST) & 5.2 TF-IDF EXTRACTION
# ==============================================================================
# Stratified split berdasarkan kombinasi label komposit 6 aspek
composite_label = y.astype(str).agg("_".join, axis=1)
stratify_col = composite_label if composite_label.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify_col
)

# Ekstraksi fitur TF-IDF (Unigram & Bigram) - Fit hanya pada X_train untuk mencegah data leakage
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("=== 5.4 DATA SPLITTING & TF-IDF EXTRACTION ===")
print(f"X_train TF-IDF shape : {X_train_tfidf.shape} (80%)")
print(f"X_test TF-IDF shape  : {X_test_tfidf.shape} (20%)")
print(f"Contoh fitur bigram  : {[f for f in tfidf.get_feature_names_out() if ' ' in f][:5]}\n")

# ==============================================================================
# 5.5 DATA BALANCING (CLASS WEIGHTS)
# ==============================================================================
print("=== 5.5 DATA BALANCING (DISTRIBUSI KELAS TRAIN) ===")
for col in target_cols:
    counts = y_train[col].value_counts().to_dict()
    print(f"{col:<18} -> Negatif(0): {counts.get(0, 0)}, Netral(1): {counts.get(1, 0)}, Positif(2): {counts.get(2, 0)}")

# Konfigurasi model Multi-Output dengan penyeimbang bobot kelas otomatis
model = MultiOutputClassifier(
    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
)
print("\nKonfigurasi Balancing: MultiOutputClassifier(LogisticRegression(class_weight='balanced')) siap digunakan.")

=== 5.1 DATA CLEANING ===
Total data bersih        : 6120 baris
Data tanpa aspek (netral): 1032 baris
Contoh hasil pembersihan teks:
                                              ulasan  \
0  Styling:Skena banget lah Compass nih\nKenyaman...   
1  Tampilan:sesuai keinginan\nWarna:Sesuai dengan...   
2  Styling:kata cowoku bagus\nKenyamanan:kata cow...   

                                        ulasan_clean  
0  skena banget lah compass nih nyaman banget emp...  
1  sesuai keinginan sesuai dengan gambar beli tan...  
2  kata cowoku bagus kata cowoku empuk sepatunya ...   

=== 5.2 & 5.3 FEATURE SELECTION & TARGET ENCODING ===
Fitur Input (X)  : ulasan_clean (shape: (6120,))
Target Output (Y): ['aspek_kualitas', 'aspek_bahan', 'aspek_ukuran', 'aspek_desain', 'aspek_warna', 'aspek_kenyamanan'] (shape: (6120, 6))
   aspek_kualitas  aspek_bahan  aspek_ukuran  aspek_desain  aspek_warna  \
0               2            1             1             2            1   
1               2           

In [8]:
# ==============================================================================
# 6.2 ADVANCED MODEL: LIGHTGBM + CLASSIFIER CHAIN + 5-FOLD CV (LENGKAP)
# ==============================================================================
import re
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import ClassifierChain
from sklearn.metrics import accuracy_score, f1_score
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)

# ------------------------------------------------------------------------------
# 1. BAB 5: LOAD DATA, CLEANING, PREPROCESSING & SPLIT (80/20)
# ------------------------------------------------------------------------------
df = pd.read_excel("/content/dataset_labeled_v6.xlsx")

target_cols = [
    "aspek_kualitas", "aspek_bahan", "aspek_ukuran",
    "aspek_desain", "aspek_warna", "aspek_kenyamanan"
]

df = df.dropna(subset=["ulasan"]).copy()
df[target_cols] = df[target_cols].fillna("netral")
df = df.drop_duplicates(subset=["ulasan"]).copy()

df["ulasan"] = df["ulasan"].astype(str)
for col in target_cols:
    df[col] = df[col].astype(str)

def preprocess_text(text):
    text = text.lower()
    text = re.sub(
        r"\b(harga|styling|tampilan|saran tampilan|kualitas|warna|bahan|ukuran|kenyamanan)\s*:",
        " ",
        text,
    )
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["ulasan_clean"] = df["ulasan"].apply(preprocess_text)
df = df[df["ulasan_clean"].str.split().str.len() > 2].reset_index(drop=True)

X = df["ulasan_clean"]
label_map = {"negatif": 0, "netral": 1, "positif": 2}
y = df[target_cols].apply(lambda col: col.str.strip().str.lower().map(label_map)).astype(int)

composite_label = y.astype(str).agg("_".join, axis=1)
stratify_col = composite_label if composite_label.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=stratify_col
)

X_tr_raw = X_train.reset_index(drop=True).astype(str)
y_tr_arr = np.asarray(y_train.reset_index(drop=True), dtype=np.int64)
X_te_raw = X_test.reset_index(drop=True).astype(str)
y_te_arr = np.asarray(y_test.reset_index(drop=True), dtype=np.int64)

# ------------------------------------------------------------------------------
# 2. FUNGSI BANTU & EVALUASI
# ------------------------------------------------------------------------------
def compute_smoothed_weights(y_arr):
    counts = np.bincount(y_arr.flatten(), minlength=3).astype(np.float32)
    inv_freq = counts.sum() / np.maximum(counts, 1.0)
    w = np.sqrt(inv_freq / inv_freq.min())
    return {0: float(w[0]), 1: float(w[1]), 2: float(w[2])}

def eval_multi_aspect(y_true, y_pred):
    accs, macro_f1s, weighted_f1s, neg_f1s, neu_f1s, pos_f1s = [], [], [], [], [], []
    for i in range(y_true.shape[1]):
        yt, yp = y_true[:, i], y_pred[:, i]
        accs.append(accuracy_score(yt, yp))
        macro_f1s.append(f1_score(yt, yp, average="macro", zero_division=0))
        weighted_f1s.append(f1_score(yt, yp, average="weighted", zero_division=0))
        f1_cls = f1_score(yt, yp, labels=[0, 1, 2], average=None, zero_division=0)
        neg_f1s.append(f1_cls[0])
        neu_f1s.append(f1_cls[1])
        pos_f1s.append(f1_cls[2])
    return {
        "accuracy": float(np.mean(accs)),
        "macro_f1": float(np.mean(macro_f1s)),
        "weighted_f1": float(np.mean(weighted_f1s)),
        "f1_neg": float(np.mean(neg_f1s)),
        "f1_neu": float(np.mean(neu_f1s)),
        "f1_pos": float(np.mean(pos_f1s)),
    }

def build_lgb_chain(cw):
    base_lgb = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        num_leaves=15,
        min_child_samples=5,
        class_weight=cw,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )
    return ClassifierChain(base_lgb, order="random", random_state=SEED)

# ------------------------------------------------------------------------------
# 3. 5-FOLD CROSS-VALIDATION LENGKAP PADA DATA TRAIN
# ------------------------------------------------------------------------------
print("=== 6.2 ADVANCED MODEL: 5-FOLD CROSS-VALIDATION (LIGHTGBM + CLASSIFIER CHAIN) ===")
t_all = time.time()
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

fold_records = []
for fold, (tr_idx, va_idx) in enumerate(kf.split(X_tr_raw), start=1):
    t_fold = time.time()
    X_f_tr, X_f_va = X_tr_raw.iloc[tr_idx], X_tr_raw.iloc[va_idx]
    y_f_tr, y_f_va = y_tr_arr[tr_idx], y_tr_arr[va_idx]

    vec_fold = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
    X_f_tr_vec = vec_fold.fit_transform(X_f_tr)
    X_f_va_vec = vec_fold.transform(X_f_va)

    cw_fold = compute_smoothed_weights(y_f_tr)
    chain_fold = build_lgb_chain(cw_fold)
    chain_fold.fit(X_f_tr_vec, y_f_tr)

    y_f_pred = chain_fold.predict(X_f_va_vec)
    fold_dur = time.time() - t_fold

    m = eval_multi_aspect(y_f_va, y_f_pred)
    m["fold"] = fold
    m["waktu"] = fold_dur
    fold_records.append(m)

    print(
        f"Fold {fold}/5 | Acc: {m['accuracy']:.4f} | Macro F1: {m['macro_f1']:.4f} | "
        f"W-F1: {m['weighted_f1']:.4f} | Neg(0): {m['f1_neg']:.4f} | "
        f"Neu(1): {m['f1_neu']:.4f} | Pos(2): {m['f1_pos']:.4f} | Waktu: {fold_dur:.2f}s"
    )

df_folds = pd.DataFrame(fold_records)
cv_total_time = time.time() - t_all

print("\n--- Rata-rata 5-Fold CV ---")
print(f"Mean Accuracy    : {df_folds['accuracy'].mean():.4f} (±{df_folds['accuracy'].std():.4f})")
print(f"Mean Macro F1    : {df_folds['macro_f1'].mean():.4f} (±{df_folds['macro_f1'].std():.4f})")
print(f"Mean Weighted F1 : {df_folds['weighted_f1'].mean():.4f} (±{df_folds['weighted_f1'].std():.4f})")
print(f"Mean F1 Neg(0)   : {df_folds['f1_neg'].mean():.4f} (±{df_folds['f1_neg'].std():.4f})")
print(f"Mean F1 Neu(1)   : {df_folds['f1_neu'].mean():.4f} (±{df_folds['f1_neu'].std():.4f})")
print(f"Mean F1 Pos(2)   : {df_folds['f1_pos'].mean():.4f} (±{df_folds['f1_pos'].std():.4f})")
print(f"Total waktu 5-Fold: {cv_total_time:.2f}s (rata-rata {df_folds['waktu'].mean():.2f}s/fold)\n")

# ------------------------------------------------------------------------------
# 4. FIT PENUH PADA X_TRAIN & EVALUASI PADA X_TEST
# ------------------------------------------------------------------------------
t_test = time.time()
tfidf_full = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
X_tr_full_vec = tfidf_full.fit_transform(X_tr_raw)
X_te_full_vec = tfidf_full.transform(X_te_raw)

cw_full = compute_smoothed_weights(y_tr_arr)
advanced_model_kf = build_lgb_chain(cw_full)
advanced_model_kf.fit(X_tr_full_vec, y_tr_arr)

y_te_pred = advanced_model_kf.predict(X_te_full_vec)
test_metrics = eval_multi_aspect(y_te_arr, y_te_pred)
test_time = time.time() - t_test

print("--- Evaluasi Test Set (20% Hold-out) ---")
print(
    f"Test Set | Acc: {test_metrics['accuracy']:.4f} | Macro F1: {test_metrics['macro_f1']:.4f} | "
    f"W-F1: {test_metrics['weighted_f1']:.4f} | F1 Neg(0): {test_metrics['f1_neg']:.4f} | "
    f"F1 Neu(1): {test_metrics['f1_neu']:.4f} | F1 Pos(2): {test_metrics['f1_pos']:.4f} | "
    f"Waktu: {test_time:.2f}s"
)

=== 6.2 ADVANCED MODEL: 5-FOLD CROSS-VALIDATION (LIGHTGBM + CLASSIFIER CHAIN) ===
Fold 1/5 | Acc: 0.9024 | Macro F1: 0.7104 | W-F1: 0.8986 | Neg(0): 0.3645 | Neu(1): 0.9354 | Pos(2): 0.8313 | Waktu: 25.77s
Fold 2/5 | Acc: 0.9030 | Macro F1: 0.7266 | W-F1: 0.9000 | Neg(0): 0.4280 | Neu(1): 0.9380 | Pos(2): 0.8140 | Waktu: 25.17s
Fold 3/5 | Acc: 0.9055 | Macro F1: 0.7087 | W-F1: 0.9025 | Neg(0): 0.3536 | Neu(1): 0.9382 | Pos(2): 0.8344 | Waktu: 24.76s
Fold 4/5 | Acc: 0.8970 | Macro F1: 0.7129 | W-F1: 0.8942 | Neg(0): 0.3954 | Neu(1): 0.9323 | Pos(2): 0.8111 | Waktu: 25.57s
Fold 5/5 | Acc: 0.8992 | Macro F1: 0.6953 | W-F1: 0.8962 | Neg(0): 0.3326 | Neu(1): 0.9337 | Pos(2): 0.8197 | Waktu: 24.98s

--- Rata-rata 5-Fold CV ---
Mean Accuracy    : 0.9014 (±0.0033)
Mean Macro F1    : 0.7108 (±0.0112)
Mean Weighted F1 : 0.8983 (±0.0032)
Mean F1 Neg(0)   : 0.3748 (±0.0374)
Mean F1 Neu(1)   : 0.9355 (±0.0026)
Mean F1 Pos(2)   : 0.8221 (±0.0103)
Total waktu 5-Fold: 126.47s (rata-rata 25.25s/fold)

